In [ ]:
from pathlib import Path
from typing import Iterable, List

import numpy as np
from PIL import Image
from IPython.display import display

from src.vibe_blending import run_vibe_blend_not_safe
from src.ipadapter_model import create_image_grid

In [ ]:
CONFIG_PATH = Path('config.yaml')
IMAGES_DIR = Path('images')

POSITIVE_IMAGE_PATHS = [
    IMAGES_DIR / 'playviolin_hr.png',
    IMAGES_DIR / 'playguitar_hr.png',
]

EXTRA_IMAGE_PATHS = [
    # ...
]

NEGATIVE_IMAGE_PATHS = [
    # ...
]

def load_images(paths: Iterable[Path]) -> List[Image.Image]:
    return [Image.open(path).convert('RGB') for path in paths]

positive_images = load_images(POSITIVE_IMAGE_PATHS)
extra_images = load_images(EXTRA_IMAGE_PATHS)
negative_images = load_images(NEGATIVE_IMAGE_PATHS)

print(f'Loaded {len(positive_images)} positive, {len(extra_images)} extra, {len(negative_images)} negative images')

In [ ]:
GRID_PREVIEW_SIZE = (256, 256)
GRID_BLEND_SIZE = (512, 512)

def resize_for_grid(images: List[Image.Image], size: tuple[int, int]) -> List[Image.Image]:
    return [img.resize(size, Image.Resampling.LANCZOS) for img in images]

def show_row(title: str, images: List[Image.Image]):
    if not images:
        return
    print(f"{title} ({len(images)} images)")
    thumbs = resize_for_grid(images, GRID_PREVIEW_SIZE)
    grid = create_image_grid(thumbs, rows=1, cols=len(thumbs))
    display(grid)

show_row('Positives', positive_images)
show_row('Extra references', extra_images)
show_row('Negatives (attributes to suppress)', negative_images)

In [ ]:
alpha_weights = np.linspace(0, 1, 10).tolist()
print(f'α values: {alpha_weights}')

blended_with_negatives = run_vibe_blend_not_safe(
    image1=positive_images[0],
    image2=positive_images[1],
    extra_images=extra_images,
    negative_images=negative_images,
    config_path=str(CONFIG_PATH),
    interpolation_weights=alpha_weights,
    n_clusters=20,
)

sequence_with_neg = [positive_images[0], *blended_with_negatives, positive_images[1]]
rows = int(np.ceil(len(sequence_with_neg) / 4))
normalized_sequence = resize_for_grid(sequence_with_neg, GRID_BLEND_SIZE)
neg_grid = create_image_grid(normalized_sequence, rows=rows, cols=4)
display(neg_grid)